# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tokihab/FlyRank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I want to find pages that show up high on Google (good position)

 and get seen a lot (high volume/impressions),

 but people just ignore them and don't click (low CTR).


Action: CHANGE_CONTENT_ARCHETYPE

(Because the content style is not matching what the user wants to click).

Reason Code: LOW_CTR_GOOD_POSITION

Checking Two Signals:

    Signal 1 (FlyRank Session Flag - CTR vs Position): I need to check if being higher on Google actually gives better CTR.

    Signal 2 (Volume): I need to check if having more impressions actually leads to more clicks normally.

In [1]:
import pandas as pd
import pyarrow.compute as pc
import datetime
import os
from datasets import load_dataset
from google.colab import userdata

print("Loading and filtering data fast...")
hf_token = userdata.get('HF_TOKEN')
ds_fact = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", split="train", token=hf_token)
df_dim = load_dataset("FlyRank/internship-warehouse", "dim_content", split="train", token=hf_token).to_pandas()

# Filter for March 2026 natively
arrow_table = ds_fact.data.table
date_filter = (pc.field("report_date") >= datetime.date(2026, 3, 1)) & (pc.field("report_date") < datetime.date(2026, 4, 1))
df_mar_daily = arrow_table.filter(date_filter).to_pandas()

# Group data by page
df_mar = df_mar_daily.groupby('content_hash_id', observed=True).agg({
    'gsc_clicks': 'sum',
    'gsc_impressions': 'sum',
    'gsc_avg_position': 'mean',
}).rename(columns={'gsc_clicks': 'clicks', 'gsc_impressions': 'impressions', 'gsc_avg_position': 'position'}).reset_index()

df_mar['ctr'] = (df_mar['clicks'] / df_mar['impressions']).fillna(0.0)

print("\n--- SIGNAL 1: Position vs CTR ---")
# Bucket the positions
bins_pos = [0, 3, 10, 100]
labels_pos = ['Top 3', 'Page 1 (4-10)', 'Page 2+']
df_mar['position_bucket'] = pd.cut(df_mar['position'], bins=bins_pos, labels=labels_pos)

sig1_table = df_mar.groupby('position_bucket', observed=True).agg(
    avg_ctr=('ctr', 'mean'),
    n_pages=('content_hash_id', 'count')
)
print(sig1_table)
print("Verdict 1: CONFIRMED (Top 3 gets the highest CTR, dropping fast after that).")

print("\n--- SIGNAL 2: Volume (Impressions) vs Clicks ---")
# Bucket the impressions
bins_imp = [0, 100, 1000, 1000000]
labels_imp = ['Low (<100)', 'Medium (100-1k)', 'High (>1k)']
df_mar['volume_bucket'] = pd.cut(df_mar['impressions'], bins=bins_imp, labels=labels_imp)

sig2_table = df_mar.groupby('volume_bucket', observed=True).agg(
    avg_clicks=('clicks', 'mean'),
    n_pages=('content_hash_id', 'count')
)
print(sig2_table)
print("Verdict 2: CONFIRMED (High volume pages get way more clicks overall).")

Loading and filtering data fast...


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 7.12MB            

...

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]


--- SIGNAL 1: Position vs CTR ---
                  avg_ctr  n_pages
position_bucket                   
Top 3            0.010589    16144
Page 1 (4-10)    0.004926    81988
Page 2+          0.002458    77070
Verdict 1: CONFIRMED (Top 3 gets the highest CTR, dropping fast after that).

--- SIGNAL 2: Volume (Impressions) vs Clicks ---
                 avg_clicks  n_pages
volume_bucket                       
Low (<100)         0.086139    75506
Medium (100-1k)    0.943894    56197
High (>1k)        16.926479    45035
Verdict 2: CONFIRMED (High volume pages get way more clicks overall).


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I will calculate an "Expected CTR" based on the page's position bucket.

Score = Impressions * (Expected CTR - Real CTR).

If the score is a big positive number, it means we are losing a lot of clicks we should be getting. I will rank them from highest lost clicks to lowest.

In [2]:
# Create Expected CTR based on Signal 1 buckets
def get_expected_ctr(pos):
    if pos <= 3: return 0.20
    elif pos <= 10: return 0.05
    else: return 0.01

df_mar['expected_ctr'] = df_mar['position'].apply(get_expected_ctr)

# Calculate the baseline score (Lost clicks)
df_mar['ctr_deficit'] = df_mar['expected_ctr'] - df_mar['ctr']
df_mar['baseline_score'] = df_mar['impressions'] * df_mar['ctr_deficit']

# Filter only pages that actually need fixing (Score > 0)
df_queue = df_mar[df_mar['baseline_score'] > 0].copy()

# Add the required Rule labels
df_queue['action_label'] = 'CHANGE_CONTENT_ARCHETYPE'
df_queue['reason_code'] = 'LOW_CTR_GOOD_POSITION'

# Rank them from highest score to lowest
df_queue = df_queue.sort_values(by='baseline_score', ascending=False)

# Keep only the important columns for the output
output_cols = ['content_hash_id', 'baseline_score', 'action_label', 'reason_code', 'impressions', 'position', 'ctr']
df_final_queue = df_queue[output_cols]

# Save to CSV
os.makedirs("work/outputs", exist_ok=True)
csv_path = "work/outputs/baseline_action_score.csv"
df_final_queue.to_csv(csv_path, index=False)

print(f"Success! Ranked queue saved to {csv_path}")
print(f"Total actionable rows found: {len(df_final_queue)}")

Success! Ranked queue saved to work/outputs/baseline_action_score.csv
Total actionable rows found: 172312


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Here I look at the top 10 pages my rule found. For each one, I will write what the action is, why it was chosen, and what could make my rule wrong (like if the page is just a contact page that doesn't need high clicks).

In [3]:
# Get the top 10 rows
top_10 = df_final_queue.head(10).reset_index()

print("--- TOP 10 REVIEW ---")
for index, row in top_10.iterrows():
    page_id = row['content_hash_id'][:12] + "..." # Shorten ID for easy reading
    score = round(row['baseline_score'], 1)
    pos = round(row['position'], 1)
    imp = row['impressions']

    print(f"\nRank {index + 1} | Page ID: {page_id} | Score: {score}")
    print(f"Action: {row['action_label']}")
    print(f"Why it is here (Reason): {row['reason_code']} - It got {imp} impressions at position {pos}, but the CTR is too low.")
    print(f"What would make it wrong: If this page ranks for a brand name that isn't ours, people will never click it no matter what archetype we use.")

--- TOP 10 REVIEW ---

Rank 1 | Page ID: content_eadb... | Score: 117756.8
Action: CHANGE_CONTENT_ARCHETYPE
Why it is here (Reason): LOW_CTR_GOOD_POSITION - It got 617124 impressions at position 2.4, but the CTR is too low.
What would make it wrong: If this page ranks for a brand name that isn't ours, people will never click it no matter what archetype we use.

Rank 2 | Page ID: content_ec2e... | Score: 47575.2
Action: CHANGE_CONTENT_ARCHETYPE
Why it is here (Reason): LOW_CTR_GOOD_POSITION - It got 245276 impressions at position 2.9, but the CTR is too low.
What would make it wrong: If this page ranks for a brand name that isn't ours, people will never click it no matter what archetype we use.

Rank 3 | Page ID: content_0e03... | Score: 43542.0
Action: CHANGE_CONTENT_ARCHETYPE
Why it is here (Reason): LOW_CTR_GOOD_POSITION - It got 221310 impressions at position 2.7, but the CTR is too low.
What would make it wrong: If this page ranks for a brand name that isn't ours, people will never

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some picks at the bottom of the list are very weak.

If a page only gets 10 impressions a month, the score will be tiny. It is not worth our time to fix an archetype for a page nobody searches for.
Also, I checked below to make sure no future data (April) leaked into my baseline.

In [4]:
print("--- LEAKAGE CHECK ---")
max_date = df_mar_daily['report_date'].max()
print(f"The newest date in the data is: {max_date}")
if max_date < datetime.date(2026, 4, 1):
    print("Leakage Check PASSED: No future data from April leaked in.")
else:
    print("WARNING: Future data found!")

print("\n--- WEAK PICKS ---")
weak_picks = df_final_queue.tail(3)
print("Here are the bottom 3 scores. They are weak because their impressions are too low to matter:")
display(weak_picks[['content_hash_id', 'baseline_score', 'impressions']])

--- LEAKAGE CHECK ---
The newest date in the data is: 2026-03-31
Leakage Check PASSED: No future data from April leaked in.

--- WEAK PICKS ---
Here are the bottom 3 scores. They are weak because their impressions are too low to matter:


,content_hash_id,baseline_score,impressions
328656,content_fde6e4160dd74fbe,0.01,1
36276,content_1c29218e5709369f,0.01,1
325173,content_fb436566f5f0506f,0.01,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.